In [12]:
import pandas as pd
from pathlib import Path

CLEAN_DATA_DIR: Path = Path("../data/cleaned")

ORDER_DATE_COLUMNS: list[str] = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Load cleaned datasets
df_orders: pd.DataFrame = pd.read_csv(
    CLEAN_DATA_DIR / "olist_orders_cleaned.csv",
    parse_dates=ORDER_DATE_COLUMNS
)

df_order_items: pd.DataFrame = pd.read_csv(
    CLEAN_DATA_DIR / "olist_order_items_cleaned.csv"
)

df_customers: pd.DataFrame = pd.read_csv(
    CLEAN_DATA_DIR / "olist_customers_cleaned.csv",
    dtype={"customer_zip_code_prefix": str}
)

df_sellers: pd.DataFrame = pd.read_csv(
    CLEAN_DATA_DIR / "olist_sellers_cleaned.csv"
)

print(f"Orders: {df_orders.shape}")
print(f"Order Items: {df_order_items.shape}")
print(f"Customers: {df_customers.shape}")
print(f"Sellers: {df_sellers.shape}")


Orders: (99441, 8)
Order Items: (112650, 7)
Customers: (99441, 5)
Sellers: (3095, 4)


In [20]:
# Total order value, freight ratio, and item counts
df_order_items["total_item_value"] = df_order_items["price"] + df_order_items["freight_value"]

df_order_items["freight_ratio"] = (df_order_items["freight_value"] / df_order_items["total_item_value"]).round(4)

order_financials: pd.DataFrame = df_order_items.groupby("order_id").agg(
    total_order_value=("total_item_value", "sum"),
    total_item_price=("price", "sum"),
    total_freight_value=("freight_value", "sum"),
    total_items_in_order=("order_item_id", "count")
).reset_index()

order_financials["total_order_value"] = order_financials["total_order_value"].round(2)
order_financials["total_item_price"] = order_financials["total_item_price"].round(2)
order_financials["total_freight_value"] = order_financials["total_freight_value"].round(2)

print(order_financials.head(5))


                           order_id  total_order_value  total_item_price  \
0  00010242fe8c5a6d1ba2dd792cb16214              72.19             58.90   
1  00018f77f2f0320c557190d7a144bdd3             259.83            239.90   
2  000229ec398224ef6ca0657da4fc703e             216.87            199.00   
3  00024acbcdf0a6daa1e931b038114c75              25.78             12.99   
4  00042b26cf59d7ce69dfabb4e55b4fd9             218.04            199.90   

   total_freight_value  total_items_in_order  
0                13.29                     1  
1                19.93                     1  
2                17.87                     1  
3                12.79                     1  
4                18.14                     1  


In [14]:
# Delivery duration in days (purchase to customer delivery)
transit_duration: pd.Series = df_orders["order_delivered_customer_date"] - df_orders["order_purchase_timestamp"]

df_orders["delivery_days"] = transit_duration.dt.total_seconds() / 86400.0
df_orders["delivery_days"] = df_orders["delivery_days"].round(1)

print(df_orders["delivery_days"].describe())


count    96476.000000
mean        12.558652
std          9.546468
min          0.500000
25%          6.800000
50%         10.200000
75%         15.700000
max        209.600000
Name: delivery_days, dtype: float64


In [15]:
# Delivery delay variance and binary delay indicator (is_delayed)
delay_duration: pd.Series = df_orders["order_delivered_customer_date"] - df_orders["order_estimated_delivery_date"]

df_orders["delivery_delay_days"] = (delay_duration.dt.total_seconds() / 86400.0).round(1)

df_orders["is_delayed"] = 0
df_orders.loc[df_orders["delivery_delay_days"] > 0, "is_delayed"] = 1

print(df_orders["is_delayed"].value_counts(normalize=True).round(4) * 100)


is_delayed
0    92.19
1     7.81
Name: proportion, dtype: float64


In [16]:
# Merge order financials and customer attributes into orders dataset
df_orders_enriched: pd.DataFrame = pd.merge(
    df_orders,
    order_financials,
    on="order_id",
    how="left"
)

customer_lookup_columns: list[str] = [
    "customer_id",
    "customer_unique_id",
    "customer_state"
]

df_orders_enriched = pd.merge(
    df_orders_enriched,
    df_customers[customer_lookup_columns],
    on="customer_id",
    how="left"
)

print(f"Enriched orders shape: {df_orders_enriched.shape}")
print(df_orders_enriched[["order_id", "customer_unique_id", "total_order_value", "delivery_days", "is_delayed"]].head(2))


Enriched orders shape: (99441, 17)
                           order_id                customer_unique_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff   
1  53cdb2fc8bc7dce0b6741e2150273451  af07308b275d755c9edb36a90c618231   

   total_order_value  delivery_days  is_delayed  
0              38.71            8.4           0  
1             141.46           13.8           0  


In [17]:
# Customer lifetime spending, AOV, and repeat purchase flag
customer_metrics: pd.DataFrame = df_orders_enriched.groupby("customer_unique_id").agg(
    customer_order_count=("order_id", "nunique"),
    customer_total_spending=("total_order_value", "sum")
).reset_index()

customer_metrics["customer_total_spending"] = customer_metrics["customer_total_spending"].round(2)

customer_metrics["average_order_value"] = (
    customer_metrics["customer_total_spending"] / customer_metrics["customer_order_count"]
).round(2)

customer_metrics["is_repeat_customer"] = 0
customer_metrics.loc[customer_metrics["customer_order_count"] > 1, "is_repeat_customer"] = 1

print(f"Total Unique Customers: {customer_metrics.shape[0]}")
print(customer_metrics["is_repeat_customer"].value_counts(normalize=True).round(4) * 100)


Total Unique Customers: 96096
is_repeat_customer
0    96.88
1     3.12
Name: proportion, dtype: float64


In [18]:
# Seller revenue and order volume metrics
seller_metrics: pd.DataFrame = df_order_items.groupby("seller_id").agg(
    seller_revenue=("price", "sum"),
    seller_order_count=("order_id", "nunique")
).reset_index()

seller_metrics["seller_revenue"] = seller_metrics["seller_revenue"].round(2)

seller_geo_columns: list[str] = ["seller_id", "seller_state", "seller_city"]

seller_metrics = pd.merge(
    seller_metrics,
    df_sellers[seller_geo_columns],
    on="seller_id",
    how="left"
)

print(seller_metrics.sort_values(by="seller_revenue", ascending=False).head(5))


                             seller_id  seller_revenue  seller_order_count  \
857   4869f7a5dfa277a7dca6462dcf3b52b2       229472.63                1132   
1013  53243585a1d6dc2643021fd1853d8905       222776.05                 358   
881   4a3ca9315b744ce9f8e9374361493884       200472.92                1806   
3024  fa1c13f2614d7b5c4749cbc52fecda94       194042.03                 585   
1535  7c67e1448b00f6e969d365cea6b010ab       187923.89                 982   

     seller_state       seller_city  
857            SP           guariba  
1013           BA  lauro de freitas  
881            SP          ibitinga  
3024           SP            sumare  
1535           SP   itaquaquecetuba  


In [21]:
# Export engineered feature datasets
ORDERS_FEATURES_PATH: Path = CLEAN_DATA_DIR / "orders_features.csv"
CUSTOMER_FEATURES_PATH: Path = CLEAN_DATA_DIR / "customer_features.csv"
SELLER_FEATURES_PATH: Path = CLEAN_DATA_DIR / "seller_features.csv"

df_orders_enriched.to_csv(ORDERS_FEATURES_PATH, index=False)
customer_metrics.to_csv(CUSTOMER_FEATURES_PATH, index=False)
seller_metrics.to_csv(SELLER_FEATURES_PATH, index=False)

print(f"Exported orders_features.csv: {ORDERS_FEATURES_PATH.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Exported customer_features.csv: {CUSTOMER_FEATURES_PATH.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Exported seller_features.csv: {SELLER_FEATURES_PATH.stat().st_size / 1024:.2f} KB")
print("All 9 domain features successfully verified and saved.")


Exported orders_features.csv: 22.44 MB
Exported customer_features.csv: 4.66 MB
Exported seller_features.csv: 173.71 KB
All 9 domain features successfully verified and saved.
